In [1]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [18]:
ativo = ['a','b','c','d','e','f']
retornos = [0.1,0.16,0.08,0.13,0.2,0.11]
roa = [0.12,0.08,0.18,0.14,0.06,0.16]
piotroski = [7,4,9,6,3,8]
liquidez = [6,9,3,7,8,4]

info = {ativo[i]:{'retorno':retornos[i],'roa':roa[i],'piotroski':piotroski[i],'liquidez':liquidez[i]} for i in range(len(ativo))}

df = pd.DataFrame(info.values(), index=info.keys())

In [19]:
df

,retorno,roa,piotroski,liquidez
a,0.10,0.12,7,6
b,0.16,0.08,4,9
c,0.08,0.18,9,3
d,0.13,0.14,6,7
e,0.20,0.06,3,8
f,0.11,0.16,8,4


In [91]:
model = pyo.ConcreteModel()

model.ativo = pyo.Set(initialize = df.index)
model.retorno = pyo.Param(model.ativo, initialize = df['retorno'])
model.roa = pyo.Param(model.ativo, initialize = df['roa'])
model.piotroski = pyo.Param(model.ativo, initialize = df['piotroski'])
model.liquidez = pyo.Param(model.ativo, initialize = df['liquidez'])
model.x = pyo.Var(model.ativo, bounds=(0,1), domain=pyo.NonNegativeReals)
model.y = pyo.Var(model.ativo, domain=pyo.Binary)
# model.d = pyo.VarList(domain=pyo.NonNegativeReals)
# 3 objetivos, 3 d's
for i in range(3):

    setattr(model,f'obj{i}_mais',pyo.Var(domain=NonNegativeReals))
    setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))

# ----------------------------------
# restricoes

def rest_card(model):
    return sum(model.y[a] for a in model.ativo) <= 3
model.fcard = pyo.Constraint(rule=rest_card)

def rest_peso(model):
    return sum(model.x[a] for a in model.ativo) == 1
model.fpeso = pyo.Constraint(rule=rest_peso)

def peso_card(model,a):
    return model.x[a] <= model.y[a]
model.fpesocard = pyo.Constraint(model.ativo, rule=peso_card)

def peso_min(model,a):
    return model.x[a] >= 0.10*model.y[a]
model.fpeso_min = pyo.Constraint(model.ativo,rule=peso_min)
def peso_max(model,a):
    return model.x[a] <= 0.50*model.y[a]
model.fpeso_max = pyo.Constraint(model.ativo,rule=peso_max)

#goals Restricoes

def g1(model):
    return sum(model.x[a]*model.roa[a]  for a in model.ativo) + model.obj0_menos - model.obj0_mais == 0.14
model.fg1 = pyo.Constraint(rule=g1)

def g2(model):
    return sum(model.x[a]*model.piotroski[a] for a in model.ativo) +model.obj1_menos - model.obj1_mais ==7
model.fg2 = pyo.Constraint(rule=g2)

def g3(model):
    return sum(model.x[a]*model.liquidez[a] for a in model.ativo) + model.obj2_menos - model.obj2_mais == 6
model.fg3 = pyo.Constraint(rule=g3)

# objetivo

def robj(model):
    return model.obj0_menos + model.obj1_menos + model.obj2_menos
model.fobj = pyo.Objective(rule=robj, sense=pyo.minimize)

# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmpx2drr468.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmphev83edn.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmphev83edn.pyomo.lp
Objective sense      : Minimize
Variables            :      18  [Nneg: 6,  Box: 6,  Binary: 6]
Objective nonzeros   :       3
Linear constraints   :      23  [Less: 19,  Equal: 4]
  Nonzeros           :      72
  RHS nonzeros       :       5

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Obj

In [92]:
model.pprint()

1 Set Declarations
    ativo : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    6 : {'a', 'b', 'c', 'd', 'e', 'f'}

4 Param Declarations
    liquidez : Size=6, Index=ativo, Domain=Any, Default=None, Mutable=False
        Key : Value
          a :     6
          b :     9
          c :     3
          d :     7
          e :     8
          f :     4
    piotroski : Size=6, Index=ativo, Domain=Any, Default=None, Mutable=False
        Key : Value
          a :     7
          b :     4
          c :     9
          d :     6
          e :     3
          f :     8
    retorno : Size=6, Index=ativo, Domain=Any, Default=None, Mutable=False
        Key : Value
          a :   0.1
          b :  0.16
          c :  0.08
          d :  0.13
          e :   0.2
          f :  0.11
    roa : Size=6, Index=ativo, Domain=Any, Default=None, Mutable=False
        Key : Value
          a :  0.12
          b :  0.08
          c :

In [93]:
model.display()

Model unknown

  Variables:
    x : Size=6, Index=ativo
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          a :     0 :   0.5 :     1 : False : False : NonNegativeReals
          b :     0 :   0.0 :     1 : False : False : NonNegativeReals
          c :     0 : 0.125 :     1 : False : False : NonNegativeReals
          d :     0 : 0.375 :     1 : False : False : NonNegativeReals
          e :     0 :   0.0 :     1 : False : False : NonNegativeReals
          f :     0 :   0.0 :     1 : False : False : NonNegativeReals
    y : Size=6, Index=ativo
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          a :     0 :   1.0 :     1 : False : False : Binary
          b :     0 :   0.0 :     1 : False : False : Binary
          c :     0 :   1.0 :     1 : False : False : Binary
          d :     0 :   1.0 :     1 : False : False : Binary
          e :     0 :   0.0 :     1 : False : False : Binary
          f :     0 :   0.0 :     1 : False : False : Binary
  

In [100]:
{pyo.value(model.x[a]) for a in model.ativo if pyo.value(model.x[a]) > 0.09 }

{0.125, 0.375, 0.5}

In [108]:
print(f"Pesos-Ativos: {[[pyo.value(model.x[a]),[f'Ativo {a}']] for a in model.ativo if pyo.value(model.x[a]) > 0.09 ]}")
print(f'ROA :{pyo.value(model.fg1)}')
print(f'Piotroski :{pyo.value(model.fg2)}')
print(f'Liquidez :{pyo.value(model.fg3)}')

Pesos-Ativos: [[0.5, ['Ativo a']], [0.125, ['Ativo c']], [0.375, ['Ativo d']]]
ROA :0.14
Piotroski :7.0
Liquidez :6.0
